In [1]:
import json
import pandas as pd
from transformers import AutoTokenizer

# Load the data the same way as the familiarity notebook
with open('../data/cuad/train_separate_questions.json') as f:
    data = json.load(f)

contracts = data['data']
print(f"Total contracts: {len(contracts)}")

ModuleNotFoundError: No module named 'pandas'

In [2]:
# We use a tokenizer to chunk by tokens not characters
# This is important because transformers have a max token limit
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

CHUNK_SIZE = 512      # max tokens per chunk
OVERLAP = 64          # overlap between chunks to avoid cutting evidence in half

def chunk_contract(contract_id, context, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """
    Split a contract context into overlapping token chunks.
    Preserves the contract_id in each chunk for traceability.
    """
    tokens = tokenizer.encode(context, add_special_tokens=False)
    chunks = []
    start = 0
    chunk_index = 0

    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens)

        chunks.append({
            "contract_id": contract_id,
            "chunk_index": chunk_index,
            "chunk_text": chunk_text,
            "token_start": start,
            "token_end": end,
            "num_tokens": len(chunk_tokens)
        })

        chunk_index += 1
        start += chunk_size - overlap  # move forward with overlap

    return chunks


NameError: name 'AutoTokenizer' is not defined

In [3]:
# Run chunking on all contracts
all_chunks = []

for contract in contracts:
    contract_id = contract['title']
    context = contract['paragraphs'][0]['context']
    chunks = chunk_contract(contract_id, context)
    all_chunks.extend(chunks)

print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunks per contract: {len(all_chunks) / len(contracts):.1f}")


NameError: name 'contracts' is not defined

In [4]:
# Convert to DataFrame for easy inspection
chunks_df = pd.DataFrame(all_chunks)
print(chunks_df.shape)
print(chunks_df.head())

NameError: name 'pd' is not defined

In [5]:
# Manually check a few examples to verify chunking looks right
print("=== CHUNK INSPECTION ===\n")

for i, row in chunks_df[chunks_df['contract_id'] == contracts[0]['title']].iterrows():
    print(f"Contract: {row['contract_id']}")
    print(f"Chunk {row['chunk_index']} | Tokens {row['token_start']} - {row['token_end']} ({row['num_tokens']} tokens)")
    print(f"Text preview: {row['chunk_text'][:200]}")
    print("-" * 60)

=== CHUNK INSPECTION ===



NameError: name 'chunks_df' is not defined

In [6]:
# Verify overlap is working - last tokens of chunk N should appear at start of chunk N+1
first_contract_chunks = chunks_df[chunks_df['contract_id'] == contracts[0]['title']].reset_index(drop=True)

chunk_0_end = first_contract_chunks.loc[0, 'chunk_text'][-100:]
chunk_1_start = first_contract_chunks.loc[1, 'chunk_text'][:100:]

print("End of chunk 0:")
print(chunk_0_end)
print("\nStart of chunk 1:")
print(chunk_1_start)
print("\nOverlap verified if text above looks similar")

NameError: name 'chunks_df' is not defined

In [7]:
# Summary stats
print(f"Total contracts: {len(contracts)}")
print(f"Total chunks: {len(chunks_df)}")
print(f"Min tokens in a chunk: {chunks_df['num_tokens'].min()}")
print(f"Max tokens in a chunk: {chunks_df['num_tokens'].max()}")
print(f"Average tokens per chunk: {chunks_df['num_tokens'].mean():.1f}")

# Save chunks to CSV for next steps
chunks_df.to_csv('../data/cuad/chunks.csv', index=False)
print("\nChunks saved to ../data/cuad/chunks.csv")

NameError: name 'contracts' is not defined